In [1]:
import pandas as pd

print("Loading data...")
df_full = pd.read_csv("../data/OnlineRetail.csv", encoding='unicode_escape')

df = df_full.sample(n=20000, random_state=42)

print("Data successfully loaded!")

print(f"Data shape: {df.shape}")
df.head()

Loading data...
Data successfully loaded!
Data shape: (20000, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
209268,555200,71459,HANGING JAM JAR T-LIGHT HOLDER,24,6/1/2011 12:05,0.85,17315.0,United Kingdom
207108,554974,21128,GOLD FISHING GNOME,4,5/27/2011 17:14,6.95,14031.0,United Kingdom
167085,550972,21086,SET/6 RED SPOTTY PAPER CUPS,4,4/21/2011 17:05,0.65,14031.0,United Kingdom
471836,576652,22812,PACK 3 BOXES CHRISTMAS PANETTONE,3,11/16/2011 10:39,1.95,17198.0,United Kingdom
115865,546157,22180,RETROSPOT LAMP,2,3/10/2011 8:40,9.95,13502.0,United Kingdom


In [2]:
# check missing values
print("Checking for missing values...")
missing_values = df.isnull().sum()
missing_values

Checking for missing values...


InvoiceNo         0
StockCode         0
Description      56
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID     5059
Country           0
dtype: int64

In [3]:
print("Starting data cleaning...")
# Drop rows with missing CustomerID,Description
df_clean = df.dropna(subset=['CustomerID', 'Description'])

# 2. Quantity සහ UnitPrice එක 0 ට වඩා වැඩි ඒවා විතරක් ගැනීම (Returns සහ නොමිලේ දුන් දේවල් අයින් කිරීම)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]


# 3. අලුත් TotalSpend (මුළු වියදම) column එකක් හැදීම (පාරිභෝගිකයා කොච්චර වියදම් කරාද බලන්න)
df_clean = df_clean.copy()
df_clean['TotalSpend'] = df_clean['Quantity'] * df_clean['UnitPrice']


print(f"Cleaned Dataset shape: {df_clean.shape}")
display(df_clean.head())


Starting data cleaning...
Cleaned Dataset shape: (14600, 9)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalSpend
209268,555200,71459,HANGING JAM JAR T-LIGHT HOLDER,24,6/1/2011 12:05,0.85,17315.0,United Kingdom,20.40
207108,554974,21128,GOLD FISHING GNOME,4,5/27/2011 17:14,6.95,14031.0,United Kingdom,27.80
167085,550972,21086,SET/6 RED SPOTTY PAPER CUPS,4,4/21/2011 17:05,0.65,14031.0,United Kingdom,2.60
471836,576652,22812,PACK 3 BOXES CHRISTMAS PANETTONE,3,11/16/2011 10:39,1.95,17198.0,United Kingdom,5.85
115865,546157,22180,RETROSPOT LAMP,2,3/10/2011 8:40,9.95,13502.0,United Kingdom,19.90


- R - Recency (මෑතකාලීන බව): අන්තිමටම බඩු ගත්තේ දවස් කීයකට කලින්ද? (අඩු දවස් ගාණක් තියෙන අය තමයි අපේ හොඳම active customers ලා).

- F - Frequency (වාර ගණන): කොච්චර වාර ගණනක් කඩේට ඇවිත් ගනුදෙනු කරලා තියෙනවද? (Invoice ගාණ).

- M - Monetary (වියදම): මුළු කාලය ඇතුළත අපේ කඩේට කොච්චර සල්ලි ප්‍රමාණයක් වියදම් කරලා තියෙනවද?

In [4]:

# 1. InvoiceDate එක අනිවාර්යයෙන්ම Pandas Timestamp එකක් බවට පත් කිරීම
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# 2. Latest date එක ගන්නෙත් Pandas වල Timedelta පාවිච්චි කරලයි (එතකොට type එක වෙනස් වෙන්නේ නෑ)
latest_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

print("Calculating RFM metrics for each customer...")

# 3. CustomerID එකෙන් Group කරලා RFM අගයන් ගණනය කිරීම
rfm = df_clean.groupby('CustomerID').agg({
    # Recency: දැන් type දෙකම සමාන නිසා ලේසියෙන්ම අඩු කරන්න පුළුවන්
    'InvoiceDate': lambda x: (latest_date - x.max()).days,

    # Frequency: අනුපිටපත් නැති (unique) Invoices ගාණ
    'InvoiceNo': 'nunique',

    # Monetary: මුළු වියදම එකතු කිරීම
    'TotalSpend': 'sum'
})

# 4. Columns වල නම් ලස්සනට වෙනස් කිරීම
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalSpend': 'Monetary'
}, inplace=True)

print(f"RFM Matrix shape (Customers count): {rfm.shape[0]}")
display(rfm.head())

Calculating RFM metrics for each customer...
RFM Matrix shape (Customers count): 3079


,Recency,Frequency,Monetary
CustomerID,,,
12347.0,2,5,169.20
12349.0,19,1,84.40
12353.0,204,1,59.70
12354.0,232,1,25.45
12355.0,214,1,25.50


### ඇයි අපි මේක Scale කරන්නේ?
අපේ RFM Table එකේ Recency සහ Frequency තියෙන්නේ 1, 10, 50 වගේ පොඩි අගයන් වලින්. ඒත් Monetary (වියදම) තියෙන්නේ 1000, 5000 වගේ ලොකු අගයන් වලින්. අපි මේක කෙලින්ම K-Means එකට දුන්නොත්, අර Monetary එකේ තියෙන ලොකු අගයන් නිසා මාදිලිය (Model) අනිත් features දෙක ගණන් ගන්නේ නැතුව ඉන්නවා. ඒ නිසා අපි StandardScaler පාවිච්චි කරලා මේ ඔක්කොම එකම මට්ටමකට ගේන්න ඕනේ.

In [7]:
# පියවර 1: Elbow Method එකෙන් හොඳම Clusters ගාණ හොයාගැනීම
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
